In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 24 — Context Compression RAG
## Compress Retrieved Context Before Answer Generation

**Stack:** Ollama · LangChain · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup with Large Documents

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Long documents that retrieve chunks with noise
docs = [Document(page_content=c, metadata={"id": i}) for i, c in enumerate([
    """Introduction to Kubernetes. Kubernetes (K8s) is an open-source container
orchestration platform. It automates deployment, scaling, and management of
containerized applications. Originally designed by Google, it's now maintained
by CNCF. The name Kubernetes comes from Greek meaning 'helmsman'. It was
released in 2014 and has since become the de facto standard for container
orchestration. Many cloud providers offer managed Kubernetes services.""",

    """Kubernetes Architecture. The control plane consists of the API server,
etcd (key-value store), scheduler, and controller manager. Worker nodes run
kubelet, kube-proxy, and container runtime. Pods are the smallest deployable
units. A pod can contain one or more containers that share networking and storage.""",

    """Kubernetes Deployments. A Deployment manages ReplicaSets and provides
declarative updates. Rolling updates allow zero-downtime updates by gradually
replacing pods. Strategy types: RollingUpdate and Recreate. You can set
maxSurge and maxUnavailable to control the update pace.""",

    """Kubernetes Services. Services provide stable networking for pods.
Types: ClusterIP (internal), NodePort (external on node ports),
LoadBalancer (cloud LB), ExternalName (DNS alias). Services use label
selectors to route traffic to matching pods.""",
])]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="compress_lab")
print(f"Indexed {len(docs)} documents")

## Step 2 — Without Compression (Full Context)

In [ ]:
query = "How do I do zero-downtime updates in Kubernetes?"
retrieved = vectorstore.similarity_search(query, k=3)
full_context = "\n\n".join([d.page_content for d in retrieved])

print(f"Full context length: {len(full_context)} chars")
print(f"Retrieved {len(retrieved)} chunks\n")

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context. Be specific."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

answer_full = qa_chain.invoke({"context": full_context, "question": query})
print(f"Answer (full context): {answer_full}")

## Step 3 — With Context Compression

In [ ]:
# Compression step: extract only relevant sentences
compress_prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract ONLY the sentences from the context that are directly
relevant to answering the question. Remove all irrelevant information.
Return the extracted sentences as-is, do not paraphrase."""),
    ("human", "Question: {question}\n\nContext: {context}\n\nRelevant sentences only:")
])
compress_chain = compress_prompt | llm | StrOutputParser()

compressed = compress_chain.invoke({"context": full_context, "question": query})
print(f"Compressed context: {len(compressed)} chars ({len(compressed)/len(full_context)*100:.0f}% of original)")
print(f"\nCompressed content:\n{compressed}")

answer_compressed = qa_chain.invoke({"context": compressed, "question": query})
print(f"\nAnswer (compressed): {answer_compressed}")

## Step 4 — Comparison: Full vs Compressed

In [ ]:
queries = [
    "What is the control plane in Kubernetes?",
    "What types of Kubernetes services exist?",
    "How do rolling updates work?",
]

for q in queries:
    retrieved = vectorstore.similarity_search(q, k=3)
    full = "\n\n".join([d.page_content for d in retrieved])
    compressed = compress_chain.invoke({"context": full, "question": q})

    print(f"\nQ: {q}")
    print(f"  Full: {len(full)} chars → Compressed: {len(compressed)} chars "
          f"({len(compressed)/max(len(full),1)*100:.0f}%)")

    ans_full = qa_chain.invoke({"context": full, "question": q})
    ans_comp = qa_chain.invoke({"context": compressed, "question": q})
    print(f"  Full answer: {ans_full[:100]}...")
    print(f"  Compressed:  {ans_comp[:100]}...")

## What You Learned
- **Context compression** removes noise before generation
- **Trade-offs:** compression adds latency but improves focus
- **Measurement:** track context reduction ratio vs answer quality